In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.fully_connected import train_model
from src.helper import make_run_dir, save_run

## Load data

In [2]:
DATA_SET = 'chro_A'

df_train = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_train.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_val.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_test.parquet'))


OUTPUT_DIR = make_run_dir()

## Hyperparameters & Features

In [3]:
BATCH_SIZE    = 4096
MAX_EPOCHS    = 100
PATIENCE      = 30
LR_PATIENCE   = 8
LR_FACTOR     = 0.3
INIT_LR       = 1e-3
ACTIVATION    = 'relu'
NEURONS        = 80
HIDDEN_LAYERS  = 3


FEATURES_3 = ['delta', 'T', 'spy_ret']
FEATURES_4 = ['delta', 'T', 'spy_ret', 'vix_lag']
TARGET      = 'd_iv'

## Analytic benchmark

In [4]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 18.6509  RMSE = 0.007016
Coefficients: a = -0.140684, b = -0.081051, c = -0.057641


## 3-Feature & 3-Feature ANN 

In [5]:
train_cfg = dict(
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=INIT_LR,
    patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
    hidden_layers=HIDDEN_LAYERS, neurons=NEURONS, activation=ACTIVATION,
    target=TARGET,
)

result_3f = train_model(df_train, df_val, df_test,features=FEATURES_3, desc="ANN 3F", **train_cfg)

result_4f = train_model(df_train, df_val, df_test, features=FEATURES_4, desc="ANN 4F", **train_cfg)

ANN 3F:   0%|          | 0/100 epochs [00:00<?, ?epoch/s]  


Test:
SSE = 35.9324  RMSE = 0.009739  Time = 56.7s


ANN 4F:   0%|          | 0/100 epochs [00:00<?, ?epoch/s]  


Test:
SSE = 81.6229  RMSE = 0.014678  Time = 57.2s


In [6]:
summary = save_run(
    run_dir=OUTPUT_DIR,
    y_test=df_test[TARGET].values,
    hw=hw,
    models={"ANN-3F": result_3f, "ANN-4F": result_4f},
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,18.650940,0.000049,0.007016,0.003653,0.000991,0.002008,0.229683,NaN,NaN,NaN
1,ANN-3F,35.932430,0.000095,0.009739,0.004282,-0.000449,0.002050,-0.484073,56.7s,-92.66%,NaN
2,ANN-4F,81.622879,0.000215,0.014678,0.004680,-0.000982,0.002290,-2.371169,57.2s,-337.63%,-127.16%
3,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,113.9s,NaN,NaN
